# 🔬 CIFAR-10 Architecture Benchmark
## Comprehensive Evaluation of CNN and Transformer Models

This notebook benchmarks a curated set of **CNN and transformer architectures** on CIFAR-10 using a mix of native CIFAR-10 checkpoints and carefully controlled transfer-learning fallbacks.

### Models Covered
| Family | Models |
|--------|--------|
| VGG | VGG16-BN |
| ResNet | ResNet-50, ResNet-152 |
| DenseNet | DenseNet-121, DenseNet-201 |
| MobileNet | MobileNetV2 |
| MNASNet | MNASNet-1.0 |
| EfficientNet | EfficientNet-B0, EfficientNetV2-S |
| RegNet | RegNetY-8GF |
| Vision Transformer | ViT-Base/16 |
| Swin Transformer | Swin-Base |
| Inception / GoogLeNet family | InceptionV3 |
| ConvNeXt | ConvNeXt-Base, ConvNeXtV2-Base |

### Weight Sources
- **chenyaofo/pytorch-cifar-models** (torch.hub): CIFAR-10–native weights for VGG16-BN and MobileNetV2
- **huyvnphan/PyTorch_CIFAR10** (official repo clone + checkpoint bundle): CIFAR-10 models loaded with the repository's own `pretrained=True` API
- **HuggingFace Hub**: explicit CIFAR-10 checkpoints for ViT-Base/16 and Swin-Base
- **huyvnphan/PyTorch_CIFAR10**: native PyTorch CIFAR-10 checkpoint for InceptionV3 via the repo's official `pretrained=True` API
- **Fine-tuning fallback**: requested architectures without public CIFAR-10 checkpoints are adapted with a short validation-selected transfer-learning protocol

### Metrics Reported
Accuracy · Balanced Accuracy · Precision · Recall · F1 · ROC-AUC · PR-AUC · MCC · Cohen's Kappa  
Confusion matrices · Per-class sensitivity/specificity · Confusion-derived error rates

### Transfer-Learning Fallback Protocol
For requested architectures without public CIFAR-10 checkpoints, the notebook uses a short but technically defensible transfer-learning protocol:
- stratified 90/10 train/validation split from the CIFAR-10 training set
- 2 epochs of head-only warm-up by default
- up to 8 epochs of full-network fine-tuning by default
- AdamW + learning-rate scheduling + label smoothing + gradient clipping
- early stopping and best-checkpoint selection using validation macro-F1, with balanced accuracy as a secondary signal


In [ ]:
# Install required packages (run once)
import subprocess, sys

packages = [
    "torch", "torchvision",
    "timm>=0.9.0",
    "transformers>=4.35.0",
    "datasets",
    "tqdm",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "pandas",
    "torchinfo",
    "fvcore",
    "huggingface_hub",
    "requests",
    "gdown"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ All packages installed.")

In [ ]:
import os, sys, json, time, math, warnings, hashlib, shutil, zipfile, subprocess
from pathlib import Path
from copy import deepcopy
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
import torchvision.models as tvm

import timm
from transformers import (
    AutoFeatureExtractor, AutoImageProcessor,
    AutoModelForImageClassification, ViTForImageClassification,
    SwinForImageClassification, ConvNextForImageClassification,
    ConvNextV2ForImageClassification,
)
from huggingface_hub import hf_hub_download
import requests

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score, average_precision_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from torchinfo import summary as torchinfo_summary

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Constants ──────────────────────────────────────────────────
CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]
NUM_CLASSES     = 10
DATA_ROOT       = Path("./data")
CACHE_DIR       = Path("./model_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HUYVNPHAN_REPO_DIR = CACHE_DIR / "PyTorch_CIFAR10"

# ── Reproducibility ────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Plot style ─────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = sns.color_palette("tab20", 20)

## 📦 1 · Dataset Loading

In [ ]:
# ── Three transform pipelines ─────────────────────────────────
# Pipeline A – 32×32  (chenyaofo CIFAR-native models)
tf_32 = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std =[0.2023, 0.1994, 0.2010]),
])

# Pipeline B – 32×32  (official huyvnphan/PyTorch_CIFAR10 models)
tf_32_huy = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std =[0.2471, 0.2435, 0.2616]),
])

# Pipeline C – 224×224  (HF / timm / torchvision fallback models)
tf_224 = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

# Training transforms (used when transfer-learning fallback is triggered)
tf_train_224 = T.Compose([
    T.RandomCrop(32, padding=4),
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class IndexedTransformDataset(torch.utils.data.Dataset):
    """Apply a transform to a selected set of indices from a base torchvision dataset."""
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[int(self.indices[idx])]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

def stratified_split_indices(targets, val_ratio=0.10, seed=42):
    """Create a reproducible stratified train/validation split."""
    rng = np.random.RandomState(seed)
    targets = np.asarray(targets)
    train_idx, val_idx = [], []

    for cls in np.unique(targets):
        cls_idx = np.where(targets == cls)[0]
        rng.shuffle(cls_idx)
        n_val = max(1, int(round(len(cls_idx) * val_ratio)))
        val_idx.extend(cls_idx[:n_val].tolist())
        train_idx.extend(cls_idx[n_val:].tolist())

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx

# Download CIFAR-10
test_ds_32     = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=tf_32)
test_ds_32_huy = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=tf_32_huy)
test_ds_224    = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=tf_224)

base_train_raw = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=None)
train_idx, val_idx = stratified_split_indices(base_train_raw.targets, val_ratio=0.10, seed=42)

train_ds_224 = IndexedTransformDataset(base_train_raw, train_idx, transform=tf_train_224)
val_ds_224   = IndexedTransformDataset(base_train_raw, val_idx, transform=tf_224)

BS = 200 if device.type == "cuda" else 64
NW = 4 if device.type == "cuda" else 0

loader_32 = DataLoader(
    test_ds_32, batch_size=BS, shuffle=False,
    num_workers=NW, pin_memory=(device.type == "cuda")
)
loader_32_huy = DataLoader(
    test_ds_32_huy, batch_size=BS, shuffle=False,
    num_workers=NW, pin_memory=(device.type == "cuda")
)
loader_224 = DataLoader(
    test_ds_224, batch_size=BS, shuffle=False,
    num_workers=NW, pin_memory=(device.type == "cuda")
)
train_loader_224 = DataLoader(
    train_ds_224, batch_size=BS, shuffle=True,
    num_workers=NW, pin_memory=(device.type == "cuda")
)
val_loader_224 = DataLoader(
    val_ds_224, batch_size=BS, shuffle=False,
    num_workers=NW, pin_memory=(device.type == "cuda")
)

print(f"Test images (32):              {len(test_ds_32)}")
print(f"Test images (32, huy style):   {len(test_ds_32_huy)}")
print(f"Test images (224):             {len(test_ds_224)}")
print(f"Train split for fallback FT:   {len(train_ds_224)}")
print(f"Validation split for FT:       {len(val_ds_224)}")
print(f"Class-balanced val ratio:      {len(val_ds_224) / len(base_train_raw):.1%}")

## 🔧 2 · Utility Functions

In [ ]:
# ── Model complexity ─────────────────────────────
def count_params(model):
    """Return (total, trainable) parameter counts in millions."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total / 1e6, trainable / 1e6

def estimate_macs(model, input_size=(1, 3, 224, 224)):
    """Estimate MACs using torchinfo (returns GMACs)."""
    try:
        s = torchinfo_summary(
            model, input_size=input_size, verbose=0,
            device=device, col_names=[]
        )
        return s.total_mult_adds / 1e9
    except Exception:
        return float("nan")

def extract_top_confusions(cm, class_names, top_n=10):
    """Return the strongest off-diagonal confusions as a small DataFrame."""
    rows = []
    for i in range(len(class_names)):
        row_total = cm[i].sum()
        if row_total == 0:
            continue
        for j in range(len(class_names)):
            if i == j:
                continue
            count = int(cm[i, j])
            if count == 0:
                continue
            rows.append({
                "True Class": class_names[i],
                "Predicted As": class_names[j],
                "Count": count,
                "Rate (%)": count / row_total * 100,
            })
    if not rows:
        return pd.DataFrame(columns=["True Class", "Predicted As", "Count", "Rate (%)"])
    return (
        pd.DataFrame(rows)
        .sort_values(["Rate (%)", "Count"], ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

def build_confusion_metrics_df(cm, class_names):
    """Create one-vs-rest confusion-derived metrics for every class."""
    rows = []
    total = cm.sum()
    for i, cls_name in enumerate(class_names):
        tp = int(cm[i, i])
        fn = int(cm[i, :].sum() - tp)
        fp = int(cm[:, i].sum() - tp)
        tn = int(total - tp - fn - fp)

        sensitivity = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
        specificity = 0.0 if (tn + fp) == 0 else tn / (tn + fp)
        precision = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
        npv = 0.0 if (tn + fn) == 0 else tn / (tn + fn)
        fpr = 0.0 if (fp + tn) == 0 else fp / (fp + tn)
        fnr = 0.0 if (fn + tp) == 0 else fn / (fn + tp)

        rows.append({
            "Class": cls_name,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Precision (%)": precision * 100,
            "Recall / Sensitivity (%)": sensitivity * 100,
            "Specificity (%)": specificity * 100,
            "NPV (%)": npv * 100,
            "FPR (%)": fpr * 100,
            "FNR (%)": fnr * 100,
            "Support": int(cm[i, :].sum()),
        })
    return pd.DataFrame(rows)

def forward_logits(model, x):
    """Always return a logits tensor regardless of model family."""
    out = model(x)
    return out.logits if hasattr(out, "logits") else out

def score_predictions(labels, preds):
    """Fast validation metrics used for checkpoint selection."""
    return {
        "accuracy": accuracy_score(labels, preds) * 100,
        "balanced_accuracy": balanced_accuracy_score(labels, preds) * 100,
        "f1_macro": precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )[2] * 100,
    }

def run_epoch(model, loader, optimizer=None, criterion=None, scaler=None, desc="epoch"):
    """
    Run one epoch.
    If optimizer is None, runs in evaluation mode and returns validation statistics.
    """
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_labels = [], []

    amp_enabled = (device.type == "cuda")
    autocast_ctx = torch.cuda.amp.autocast if amp_enabled else nullcontext

    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with autocast_ctx() if not amp_enabled else autocast_ctx(enabled=True):
            logits = forward_logits(model, imgs)
            loss = criterion(logits, labels) if criterion is not None else None

        if is_train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        total_loss += (0.0 if loss is None else loss.item() * labels.size(0))
        preds = logits.argmax(1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    all_preds = np.asarray(all_preds)
    all_labels = np.asarray(all_labels)
    stats = score_predictions(all_labels, all_preds)
    stats["loss"] = total_loss / max(1, len(loader.dataset))
    return stats

# ── Evaluation ──────────────────────────────
@torch.no_grad()
def evaluate(model, loader, model_name="model", desc=None):
    """Scientific multiclass evaluation focused on classification quality."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for imgs, labels in tqdm(loader, desc=desc or f"Eval {model_name}", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        logits = forward_logits(model, imgs)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    all_preds = np.asarray(all_preds)
    all_labels = np.asarray(all_labels)
    all_probs = np.asarray(all_probs)

    y_true_bin = label_binarize(all_labels, classes=np.arange(NUM_CLASSES))
    cm = confusion_matrix(all_labels, all_preds, labels=np.arange(NUM_CLASSES))
    cm_norm = cm.astype(np.float64) / np.clip(cm.sum(axis=1, keepdims=True), a_min=1, a_max=None) * 100.0

    acc = accuracy_score(all_labels, all_preds) * 100
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100

    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="micro", zero_division=0
    )

    roc_auc_macro = roc_auc_score(y_true_bin, all_probs, multi_class="ovr", average="macro") * 100
    roc_auc_weighted = roc_auc_score(y_true_bin, all_probs, multi_class="ovr", average="weighted") * 100
    pr_auc_macro = average_precision_score(y_true_bin, all_probs, average="macro") * 100
    pr_auc_weighted = average_precision_score(y_true_bin, all_probs, average="weighted") * 100

    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)

    cls_prec, cls_rec, cls_f1, cls_support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=np.arange(NUM_CLASSES), zero_division=0
    )

    confusion_metrics_df = build_confusion_metrics_df(cm, CIFAR10_CLASSES)
    per_class_rows = []
    for i, cls_name in enumerate(CIFAR10_CLASSES):
        per_class_rows.append({
            "Class": cls_name,
            "Precision (%)": cls_prec[i] * 100,
            "Recall / Sensitivity (%)": cls_rec[i] * 100,
            "Specificity (%)": float(confusion_metrics_df.loc[i, "Specificity (%)"]),
            "NPV (%)": float(confusion_metrics_df.loc[i, "NPV (%)"]),
            "FPR (%)": float(confusion_metrics_df.loc[i, "FPR (%)"]),
            "FNR (%)": float(confusion_metrics_df.loc[i, "FNR (%)"]),
            "F1 Score (%)": cls_f1[i] * 100,
            "Support": int(cls_support[i]),
        })

    per_class_df = pd.DataFrame(per_class_rows)
    class_report = classification_report(
        all_labels, all_preds, target_names=CIFAR10_CLASSES, digits=4, zero_division=0
    )

    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), all_probs.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro) * 100

    return dict(
        accuracy=acc,
        balanced_accuracy=bal_acc,
        precision_macro=prec_macro * 100,
        recall_macro=rec_macro * 100,
        f1_macro=f1_macro * 100,
        precision_weighted=prec_weighted * 100,
        recall_weighted=rec_weighted * 100,
        f1_weighted=f1_weighted * 100,
        precision_micro=prec_micro * 100,
        recall_micro=rec_micro * 100,
        f1_micro=f1_micro * 100,
        specificity_macro=confusion_metrics_df["Specificity (%)"].mean(),
        npv_macro=confusion_metrics_df["NPV (%)"].mean(),
        fpr_macro=confusion_metrics_df["FPR (%)"].mean(),
        fnr_macro=confusion_metrics_df["FNR (%)"].mean(),
        roc_auc_macro_ovr=roc_auc_macro,
        roc_auc_weighted_ovr=roc_auc_weighted,
        roc_auc_micro=roc_auc_micro,
        pr_auc_macro=pr_auc_macro,
        pr_auc_weighted=pr_auc_weighted,
        mcc=mcc,
        cohen_kappa=kappa,
        confusion_matrix=cm,
        confusion_matrix_norm=cm_norm,
        confusion_metrics=confusion_metrics_df,
        per_class_metrics=per_class_df,
        class_report=class_report,
        top_confusions=extract_top_confusions(cm, CIFAR10_CLASSES, top_n=10),
        preds=all_preds,
        labels=all_labels,
        probs=all_probs,
        y_true_bin=y_true_bin,
        roc_curve_micro=(fpr_micro, tpr_micro),
    )

def set_trainable_scope(model, head_only=False):
    """Either train only the classifier head or the full network."""
    for p in model.parameters():
        p.requires_grad_(True)

    if not head_only:
        return

    head_keywords = ("classifier", "fc", "head", "heads")
    for name, p in model.named_parameters():
        if not any(k in name.lower() for k in head_keywords):
            p.requires_grad_(False)

# ── Fine-tuning (scientific short protocol for fallback models) ──

def infer_model_family(model):
    name = model.__class__.__name__.lower()
    if any(k in name for k in ["vit", "deit", "beit"]):
        return "vit"
    if "swin" in name:
        return "swin"
    if "convnextv2" in name:
        return "convnextv2"
    if "convnext" in name:
        return "convnext"
    if "mnasnet" in name:
        return "mnasnet"
    if "efficientnetv2" in name:
        return "efficientnetv2"
    if "efficientnet" in name:
        return "efficientnet"
    if "regnet" in name:
        return "regnet"
    if "densenet" in name:
        return "densenet"
    if "resnet" in name:
        return "resnet"
    return "generic"

def finetune(
    model,
    head_epochs=2,
    full_epochs=8,
    lr_head=1e-3,
    lr_full=1e-4,
    weight_decay=5e-5,
    patience=3,
    model_name="model",
    family=None,
):
    """
    Scientifically safer transfer-learning protocol for CIFAR-10:
      1) stratified 90/10 train/val split
      2) multi-epoch head warm-up
      3) low-LR full-network fine-tuning
      4) validation-selected checkpoint by macro-F1 + balanced accuracy
      5) model-family-specific defaults to avoid catastrophic forgetting
    """
    family = family or infer_model_family(model)

    family_cfg = {
        "vit":        dict(head_epochs=max(head_epochs, 3), full_epochs=max(full_epochs, 12), lr_head=min(lr_head, 8e-4), lr_full=min(lr_full, 5e-5), weight_decay=max(weight_decay, 1e-4), patience=max(patience, 4)),
        "swin":       dict(head_epochs=max(head_epochs, 3), full_epochs=max(full_epochs, 12), lr_head=min(lr_head, 8e-4), lr_full=min(lr_full, 5e-5), weight_decay=max(weight_decay, 1e-4), patience=max(patience, 4)),
        "convnext":   dict(head_epochs=max(head_epochs, 3), full_epochs=max(full_epochs, 12), lr_head=min(lr_head, 1e-3), lr_full=min(lr_full, 7e-5), weight_decay=max(weight_decay, 1e-4), patience=max(patience, 4)),
        "convnextv2": dict(head_epochs=max(head_epochs, 3), full_epochs=max(full_epochs, 12), lr_head=min(lr_head, 1e-3), lr_full=min(lr_full, 7e-5), weight_decay=max(weight_decay, 1e-4), patience=max(patience, 4)),
        "mnasnet":    dict(head_epochs=max(head_epochs, 3), full_epochs=max(full_epochs, 10), lr_head=min(lr_head, 1e-3), lr_full=min(lr_full, 8e-5), weight_decay=max(weight_decay, 1e-4), patience=max(patience, 4)),
        "generic":    dict(head_epochs=head_epochs, full_epochs=full_epochs, lr_head=lr_head, lr_full=lr_full, weight_decay=weight_decay, patience=patience),
    }
    cfg = family_cfg.get(family, family_cfg["generic"])
    head_epochs = cfg["head_epochs"]
    full_epochs = cfg["full_epochs"]
    lr_head = cfg["lr_head"]
    lr_full = cfg["lr_full"]
    weight_decay = cfg["weight_decay"]
    patience = cfg["patience"]

    cache_path = CACHE_DIR / f"{model_name}_scientific_ft_v3.pth"
    if cache_path.exists():
        print(f"  Loading cached scientific fine-tuned weights: {cache_path}")
        ckpt = torch.load(cache_path, map_location=device)
        if isinstance(ckpt, dict) and "state_dict" in ckpt:
            model.load_state_dict(ckpt["state_dict"])
        else:
            model.load_state_dict(ckpt)
        model.eval()
        return model

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_state = deepcopy(model.state_dict())
    best_score = -float("inf")
    history = []

    stage_plan = [
        ("head-only warm-up", head_epochs, True, lr_head),
        ("full fine-tuning", full_epochs, False, lr_full),
    ]

    print(
        f"  Scientific transfer-learning protocol for {model_name} [{family}]: "
        f"{head_epochs} head-only epoch(s) + up to {full_epochs} full epoch(s), "
        f"early stopping patience={patience}, "
        f"lr_head={lr_head:.1e}, lr_full={lr_full:.1e}"
    )

    for stage_name, n_epochs, head_only, lr in stage_plan:
        if n_epochs <= 0:
            continue

        set_trainable_scope(model, head_only=head_only)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=1, min_lr=1e-6
        )
        stale_epochs = 0

        for ep in range(1, n_epochs + 1):
            tr = run_epoch(
                model, train_loader_224,
                optimizer=optimizer,
                criterion=criterion,
                scaler=scaler,
                desc=f"{model_name} | {stage_name} | train {ep}/{n_epochs}",
            )
            va = run_epoch(
                model, val_loader_224,
                optimizer=None,
                criterion=criterion,
                scaler=scaler,
                desc=f"{model_name} | {stage_name} | val {ep}/{n_epochs}",
            )

            composite_score = 0.7 * va["f1_macro"] + 0.3 * va["balanced_accuracy"]
            improved = composite_score > (best_score + 1e-5)

            history.append({
                "stage": stage_name,
                "epoch_in_stage": ep,
                "train_loss": tr["loss"],
                "train_acc": tr["accuracy"],
                "train_f1_macro": tr["f1_macro"],
                "val_loss": va["loss"],
                "val_acc": va["accuracy"],
                "val_balanced_accuracy": va["balanced_accuracy"],
                "val_f1_macro": va["f1_macro"],
                "selection_score": composite_score,
                "lr": optimizer.param_groups[0]["lr"],
            })

            print(
                f"  [{stage_name:18s}] "
                f"ep {ep:02d}/{n_epochs} | "
                f"train loss={tr['loss']:.4f}, acc={tr['accuracy']:.2f}%, F1m={tr['f1_macro']:.2f}% | "
                f"val loss={va['loss']:.4f}, acc={va['accuracy']:.2f}%, "
                f"bal_acc={va['balanced_accuracy']:.2f}%, F1m={va['f1_macro']:.2f}%"
            )

            if improved:
                best_score = composite_score
                best_state = deepcopy(model.state_dict())
                stale_epochs = 0
            else:
                stale_epochs += 1

            scheduler.step(composite_score)

            if stale_epochs >= patience:
                print(f"  Early stopping triggered in {stage_name}.")
                break

    model.load_state_dict(best_state)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "history": history,
            "protocol": {
                "family": family,
                "head_epochs": head_epochs,
                "full_epochs": full_epochs,
                "lr_head": lr_head,
                "lr_full": lr_full,
                "weight_decay": weight_decay,
                "patience": patience,
                "selection_metric": "0.7 * val_macro_F1 + 0.3 * val_balanced_accuracy",
            },
        },
        cache_path,
    )
    print(f"  ✅ Saved best validation-selected checkpoint to {cache_path}")
    model.eval()
    return model

# ── Classifier replacement ──────────────────────────
def replace_head(model, num_classes=10):
    """Replace the final classification head for num_classes output."""
    if hasattr(model, "reset_classifier"):
        model.reset_classifier(num_classes)
        return model
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    if hasattr(model, "classifier"):
        c = model.classifier
        if isinstance(c, nn.Linear):
            model.classifier = nn.Linear(c.in_features, num_classes)
            return model
        if isinstance(c, nn.Sequential):
            layers = list(c.children())
            for idx in range(len(layers) - 1, -1, -1):
                if isinstance(layers[idx], nn.Linear):
                    layers[idx] = nn.Linear(layers[idx].in_features, num_classes)
                    model.classifier = nn.Sequential(*layers)
                    return model
    if hasattr(model, "head") and isinstance(model.head, nn.Linear):
        model.head = nn.Linear(model.head.in_features, num_classes)
        return model
    if hasattr(model, "heads") and hasattr(model.heads, "head") and isinstance(model.heads.head, nn.Linear):
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
        return model
    raise ValueError(f"Could not replace classifier head for model type: {type(model)}")

## 🤖 3 · Model Loading

### 3.1 Model Registry

In [ ]:
# We will populate this dict after loading each model.
# Structure: { display_name: { "model": nn.Module, "loader": "32" | "224",
#                               "source": str, "params_M": float,
#                               "macs_G": float } }
MODEL_REGISTRY = {}

def register(name, model, loader_key, source, input_hw=224):
    """Register an evaluated model into the registry."""
    p, _ = count_params(model)
    macs = estimate_macs(model, input_size=(1,3,input_hw,input_hw))
    MODEL_REGISTRY[name] = dict(
        model=model, loader=loader_key, source=source,
        params_M=round(p,2), macs_G=round(macs,3)
    )
    print(f"  ✅ {name:40s} | {p:6.1f}M params | {macs:.2f} GMACs | src: {source}")

print("Registry ready.")


### 3.2 chenyaofo/pytorch-cifar-models  *(CIFAR-10 Native, 32×32)*

In [ ]:
# ── Load via torch.hub ─────────────────────────────────────────
CHENYAOFO_REPO = "chenyaofo/pytorch-cifar-models"
CHENYAOFO_SRC  = "torch.hub / chenyaofo (CIFAR-10 native)"

chenyaofo_models = {
    "VGG16-BN"         : "cifar10_vgg16_bn",
    "MobileNetV2-x1.0" : "cifar10_mobilenetv2_x1_0",
    # Bonus CIFAR-native ResNets (smaller, extremely fast)
    # "ResNet-20 (CIFAR)" : "cifar10_resnet20",
    # "ResNet-56 (CIFAR)" : "cifar10_resnet56",
    # "RepVGG-A2 (CIFAR)": "cifar10_repvgg_a2",
}

for display_name, hub_name in chenyaofo_models.items():
    try:
        m = torch.hub.load(CHENYAOFO_REPO, hub_name,
                           pretrained=True, trust_repo=True, verbose=False)
        m = m.to(device).eval()
        register(display_name, m, "32", CHENYAOFO_SRC, input_hw=32)
    except Exception as e:
        print(f"  ❌ {display_name}: {e}")


### 3.3 huyvnphan/PyTorch_CIFAR10  *(Official Repo Workflow, 32×32)*

This section follows the repository's own workflow:
1. clone `huyvnphan/PyTorch_CIFAR10`
2. download the checkpoint bundle with `gdown`
3. import models from `cifar10_models.*`
4. load checkpoints with `pretrained=True`

These models use the repo's CIFAR-10 normalization:
`mean = [0.4914, 0.4822, 0.4465]`, `std = [0.2471, 0.2435, 0.2616]`.

Per the current experiment design, **ResNet-18 is intentionally excluded** from the benchmark even though the repo provides it.

In this revision, **InceptionV3 is sourced from this PyTorch-native CIFAR-10 repo**, not from the previously attempted non-PyTorch Hugging Face repo. If the native `pretrained=True` load fails, the notebook later falls back to short scientific fine-tuning for InceptionV3 just like the other missing-checkpoint models.


In [ ]:
# ── Official huyvnphan/PyTorch_CIFAR10 repo workflow (gdown version) ──

import sys
import shutil
import zipfile
import subprocess
from pathlib import Path

import requests
from tqdm.auto import tqdm

# Install gdown if missing
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

HUYVNPHAN_REPO_URL = "https://github.com/huyvnphan/PyTorch_CIFAR10.git"
HUYVNPHAN_REPO_ZIP_URL = "https://github.com/huyvnphan/PyTorch_CIFAR10/archive/refs/heads/master.zip"
HUYVNPHAN_SRC = "huyvnphan/PyTorch_CIFAR10 (official repo + gdown weights)"
HUYVNPHAN_GDRIVE_FILE_ID = "1PR1vNXH2yZlXTXSvKyXs1Uo2j4hbTdRw"

def stream_download(url, dst_path, chunk_size=2**20):
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(dst_path, "wb") as f:
            pbar = tqdm(total=total, unit="B", unit_scale=True, desc=dst_path.name) if total > 0 else None
            for chunk in r.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                if pbar is not None:
                    pbar.update(len(chunk))
            if pbar is not None:
                pbar.close()

    return dst_path

def ensure_huyvnphan_repo():
    if HUYVNPHAN_REPO_DIR.exists():
        print(f"  ✅ Repo already present: {HUYVNPHAN_REPO_DIR}")
    else:
        try:
            print("  Cloning huyvnphan/PyTorch_CIFAR10 …")
            subprocess.check_call([
                "git", "clone", "--depth", "1",
                HUYVNPHAN_REPO_URL,
                str(HUYVNPHAN_REPO_DIR),
            ])
        except Exception as e:
            print(f"  git clone failed ({e}); downloading repository zip instead …")
            repo_zip = CACHE_DIR / "PyTorch_CIFAR10_master.zip"
            stream_download(HUYVNPHAN_REPO_ZIP_URL, repo_zip)

            with zipfile.ZipFile(repo_zip, "r") as zf:
                zf.extractall(CACHE_DIR)

            extracted = CACHE_DIR / "PyTorch_CIFAR10-master"
            if HUYVNPHAN_REPO_DIR.exists():
                shutil.rmtree(HUYVNPHAN_REPO_DIR)
            extracted.rename(HUYVNPHAN_REPO_DIR)

    repo_path = str(HUYVNPHAN_REPO_DIR.resolve())
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

def _fix_state_dict_layout(target_dir):
    """
    Makes sure weights end up in:
      <repo>/cifar10_models/state_dicts/*.pt
    even if the zip extracts into a nested folder.
    """
    target_dir = Path(target_dir)
    expected_dir = target_dir / "state_dicts"

    if expected_dir.exists() and any(expected_dir.glob("*.pt")):
        return expected_dir

    pt_files = list(target_dir.rglob("*.pt"))
    if not pt_files:
        return None

    expected_dir.mkdir(parents=True, exist_ok=True)

    for pt in pt_files:
        dst = expected_dir / pt.name
        if pt.resolve() != dst.resolve():
            shutil.copy2(pt, dst)

    return expected_dir if any(expected_dir.glob("*.pt")) else None

def ensure_huyvnphan_weights():
    state_dict_dir = HUYVNPHAN_REPO_DIR / "cifar10_models" / "state_dicts"
    if state_dict_dir.exists() and any(state_dict_dir.glob("*.pt")):
        print(f"  ✅ Weights already present in {state_dict_dir}")
        return state_dict_dir

    weights_zip = CACHE_DIR / "huyvnphan_state_dicts.zip"
    print("  Downloading official weights bundle with gdown …")

    try:
        url = f"https://drive.google.com/uc?id={HUYVNPHAN_GDRIVE_FILE_ID}"
        gdown.download(url, str(weights_zip), quiet=False, fuzzy=True)
    except Exception as e:
        print(f"  ❌ Failed to download huyvnphan/PyTorch_CIFAR10 weights with gdown: {e}")
        print("  Models from this source will be skipped.")
        return None

    if not weights_zip.exists():
        print("  ❌ Weights zip was not created.")
        print("  Models from this source will be skipped.")
        return None

    actual_size = weights_zip.stat().st_size
    if actual_size < 5 * 1024 * 1024:
        print(f"  ❌ Downloaded file is unexpectedly small ({actual_size / (1024*1024):.2f} MB).")
        print("  Models from this source will be skipped.")
        return None

    target_dir = HUYVNPHAN_REPO_DIR / "cifar10_models"
    target_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(weights_zip, "r") as zf:
            zf.extractall(target_dir)
    except zipfile.BadZipFile:
        print(f"  ❌ Downloaded file {weights_zip.name} is not a valid zip file.")
        print("  Models from this source will be skipped.")
        return None

    fixed_dir = _fix_state_dict_layout(target_dir)
    if fixed_dir is None:
        print(f"  ❌ Expected extracted weights under {state_dict_dir}, but none were found.")
        print("  Models from this source will be skipped.")
        return None

    print(f"  ✅ Extracted weights to {fixed_dir}")
    return fixed_dir

ensure_huyvnphan_repo()
weights_loaded = ensure_huyvnphan_weights()

if weights_loaded:
    from cifar10_models.resnet import resnet34, resnet50
    from cifar10_models.densenet import densenet121, densenet169
    from cifar10_models.mobilenetv2 import mobilenet_v2
    from cifar10_models.googlenet import googlenet
    from cifar10_models.inception import inception_v3

    huyvnphan_specs = [
        ("ResNet-34",    resnet34),
        ("ResNet-50",    resnet50),
        ("DenseNet-121", densenet121),
        ("DenseNet-169", densenet169),
        ("MobileNetV2",  mobilenet_v2),
        ("GoogLeNet",    googlenet),
        ("InceptionV3",  inception_v3),
    ]

    for display_name, model_fn in huyvnphan_specs:
        try:
            m = model_fn(pretrained=True).to(device).eval()
            register(display_name, m, "32_huy", HUYVNPHAN_SRC, input_hw=32)
            print(f"  ✅ Loaded {display_name}")
        except Exception as e:
            print(f"  ❌ {display_name}: {e}")
else:
    print("  Skipping loading models from huyvnphan/PyTorch_CIFAR10 due to download issues.")

### 3.4 HuggingFace Hub  *(Explicit CIFAR-10 Checkpoints)*

This section only trusts checkpoints that are clearly labeled and structured as CIFAR-10 image-classification models.

The revised notebook now treats the Hugging Face models as a **Transformers-compatible subset only**:

- `aaraki/vit-base-patch16-224-in21k-finetuned-cifar10`
- `Weili/swin-base-patch4-window7-224-in22k-finetuned-cifar10`

A previously attempted Hugging Face source for InceptionV3 was removed from the unified PyTorch benchmark because it was not packaged as a standard PyTorch / Transformers checkpoint.  
Accordingly, **InceptionV3 is now handled through the PyTorch-native `huyvnphan/PyTorch_CIFAR10` workflow**, with a later fine-tuning fallback if that native checkpoint is unavailable.

This keeps the benchmark internally consistent by avoiding framework-mismatched checkpoint loading.


In [ ]:
# ── Hugging Face models: explicit CIFAR-10 Transformers checkpoints only ──
HF_CIFAR10_MODELS = {
    "ViT-Base/16": {
        "hf_id": "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10",
        "loader": "transformers",
    },
    "Swin-Base": {
        "hf_id": "Weili/swin-base-patch4-window7-224-in22k-finetuned-cifar10",
        "loader": "transformers",
    },
}

HF_TRANSFER_CANDIDATES = {
    "ConvNeXt-Base": "facebook/convnext-base-224",
}

HF_SRC = "HuggingFace Hub (explicit CIFAR-10 checkpoint)"

def load_hf_model(hf_id, display_name, loader_type="transformers"):
    """
    Load a Hugging Face-hosted CIFAR-10 checkpoint using the appropriate backend.
    In this notebook revision, the Hugging Face path is restricted to Transformers-compatible
    CIFAR-10 checkpoints only.
    """
    try:
        if loader_type == "transformers":
            model = AutoModelForImageClassification.from_pretrained(hf_id)
            return model.to(device).eval()

        raise ValueError(f"Unsupported HF loader type: {loader_type}")

    except Exception as e:
        print(f"  ❌ {display_name} ({hf_id}): {e}")
        return None

for display_name, spec in HF_CIFAR10_MODELS.items():
    print(f"Loading trusted CIFAR-10 checkpoint for {display_name} from {spec['hf_id']} …")
    model = load_hf_model(spec["hf_id"], display_name, spec.get("loader", "transformers"))
    if model is not None:
        register(display_name, model, "224", HF_SRC, input_hw=224)


### 3.5 torchvision / timm  *(Fallback Transfer Learning for Requested Models Without Public CIFAR-10 Checkpoints)*

These fallback models now use a short, technically defensible protocol rather than a single training epoch:
- 90/10 stratified train/validation split from the CIFAR-10 training set
- 2 epochs of classifier-head warm-up by default
- up to 8 epochs of full-network fine-tuning by default
- early stopping with best-checkpoint selection based on validation macro-F1 and balanced accuracy

In this revision, **ViT-Small/16 has been removed**, **Swin-Base** is handled through an explicit CIFAR-10 Hugging Face checkpoint, and **InceptionV3 only falls back to fine-tuning if its PyTorch-native CIFAR-10 checkpoint from `huyvnphan/PyTorch_CIFAR10` is unavailable**.


In [ ]:
# ── Models to fine-tune (only if not already in registry) ─────
SCIENTIFIC_FT_CFG = dict(
    head_epochs=2,
    full_epochs=8,
    patience=3,
    lr_head=1e-3,
    lr_full=1e-4,
    weight_decay=5e-5,
)

def load_or_finetune(display_name, load_fn, ft_cfg=SCIENTIFIC_FT_CFG, family=None):
    """Load model, replace the head, and apply the safer scientific fallback protocol."""
    if display_name in MODEL_REGISTRY:
        print(f"  ℹ {display_name} already loaded, skipping.")
        return

    model = load_fn()
    replace_head(model, NUM_CLASSES)
    model = model.to(device)

    model = finetune(
        model,
        model_name=display_name.replace("/", "-").replace(" ", "_"),
        family=family,
        **ft_cfg,
    )

    fam_label = family or infer_model_family(model)
    register(
        display_name,
        model,
        "224",
        f"transfer-learned on CIFAR-10 with safer family-aware protocol [{fam_label}] "
        "(multi-epoch head warm-up + low-LR full fine-tuning + early stopping + best val checkpoint)",
    )

# ResNet-152
load_or_finetune(
    "ResNet-152",
    lambda: tvm.resnet152(weights=tvm.ResNet152_Weights.IMAGENET1K_V2),
    family="resnet",
)

# DenseNet-201
load_or_finetune(
    "DenseNet-201",
    lambda: tvm.densenet201(weights=tvm.DenseNet201_Weights.IMAGENET1K_V1),
    family="densenet",
)

# MNASNet-1.0
load_or_finetune(
    "MNASNet-1.0",
    lambda: tvm.mnasnet1_0(weights=tvm.MNASNet1_0_Weights.IMAGENET1K_V1),
    family="mnasnet",
)

# EfficientNet-B0
load_or_finetune(
    "EfficientNet-B0",
    lambda: tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1),
    family="efficientnet",
)

# EfficientNetV2-S
load_or_finetune(
    "EfficientNetV2-S",
    lambda: tvm.efficientnet_v2_s(weights=tvm.EfficientNet_V2_S_Weights.IMAGENET1K_V1),
    family="efficientnetv2",
)

# RegNetY-8GF
load_or_finetune(
    "RegNetY-8GF",
    lambda: tvm.regnet_y_8gf(weights=tvm.RegNet_Y_8GF_Weights.IMAGENET1K_V2),
    family="regnet",
)

# InceptionV3 — only fine-tune if the native PyTorch CIFAR-10 checkpoint was not loaded above
load_or_finetune(
    "InceptionV3",
    lambda: tvm.inception_v3(weights=tvm.Inception_V3_Weights.IMAGENET1K_V1, aux_logits=False),
    family="inception",
)

# ConvNeXt-Base — force fine-tuning; do NOT trust generic 10-class heads
load_or_finetune(
    "ConvNeXt-Base",
    lambda: tvm.convnext_base(weights=tvm.ConvNeXt_Base_Weights.IMAGENET1K_V1),
    family="convnext",
)

# ConvNeXtV2-Base
load_or_finetune(
    "ConvNeXtV2-Base",
    lambda: timm.create_model("convnextv2_base", pretrained=True, num_classes=NUM_CLASSES),
    family="convnextv2",
)

print(f"\n{'=' * 70}")
print(f"Total models in registry: {len(MODEL_REGISTRY)}")
for k in MODEL_REGISTRY:
    print(f"  • {k}")


In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ft_models = {}
for name, info in MODEL_REGISTRY.items():
    if "transfer-learned" in info["source"]:
        ft_models[name] = info

if not ft_models:
    print("No models underwent scientific transfer-learning. Skipping visualization.")
else:
    print(f"Found {len(ft_models)} models that underwent scientific transfer-learning. Generating plots.")

    for model_name, info in ft_models.items():
        cache_path = CACHE_DIR / f"{model_name.replace('/', '-').replace(' ', '_')}_scientific_ft_v3.pth"
        if not cache_path.exists():
            print(f"  ⚠️ History for {model_name} not found at {cache_path}. Skipping.")
            continue

        try:
            ckpt = torch.load(cache_path, map_location="cpu")
            history = pd.DataFrame(ckpt["history"])

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            fig.suptitle(f"Fine-tuning History: {model_name}", fontsize=16, fontweight="bold", y=1.02)

            # --- Plot Loss ---
            # Removed explicit label='...' to prevent crashes with hue="stage"
            sns.lineplot(data=history, x="epoch_in_stage", y="train_loss", hue="stage", marker='o', ax=axes[0])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_loss", hue="stage", marker='x', linestyle='--', ax=axes[0])
            axes[0].set_title("Loss")
            axes[0].set_xlabel("Epoch in Stage")
            axes[0].set_ylabel("Loss")
            axes[0].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            # --- Plot Accuracy ---
            sns.lineplot(data=history, x="epoch_in_stage", y="train_acc", hue="stage", marker='o', ax=axes[1])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_acc", hue="stage", marker='x', linestyle='--', ax=axes[1])
            axes[1].set_title("Accuracy (%)")
            axes[1].set_xlabel("Epoch in Stage")
            axes[1].set_ylabel("Accuracy (%)")
            axes[1].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[1].get_legend_handles_labels()
            axes[1].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            # --- Plot F1 Macro ---
            sns.lineplot(data=history, x="epoch_in_stage", y="train_f1_macro", hue="stage", marker='o', ax=axes[2])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_f1_macro", hue="stage", marker='x', linestyle='--', ax=axes[2])
            axes[2].set_title("F1 Macro (%)")
            axes[2].set_xlabel("Epoch in Stage")
            axes[2].set_ylabel("F1 Macro (%)")
            axes[2].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[2].get_legend_handles_labels()
            axes[2].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plot_fname = f"finetuning_history_{model_name.replace('/', '-').replace(' ', '_')}.png"
            plt.savefig(plot_fname, dpi=150, bbox_inches="tight")
            plt.show()
            print(f"  Saved: {plot_fname}")

        except Exception as e:
            print(f"  ❌ Error plotting history for {model_name}: {e}")

In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ft_models = {}
for name, info in MODEL_REGISTRY.items():
    if "transfer-learned" in info["source"]:
        ft_models[name] = info

if not ft_models:
    print("No models underwent scientific transfer-learning. Skipping visualization.")
else:
    print(f"Found {len(ft_models)} models that underwent scientific transfer-learning. Generating plots.")

    for model_name, info in ft_models.items():
        cache_path = CACHE_DIR / f"{model_name.replace('/', '-').replace(' ', '_')}_scientific_ft_v3.pth"
        if not cache_path.exists():
            print(f"  ⚠️ History for {model_name} not found at {cache_path}. Skipping.")
            continue

        try:
            ckpt = torch.load(cache_path, map_location="cpu")
            history = pd.DataFrame(ckpt["history"])

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            fig.suptitle(f"Fine-tuning History: {model_name}", fontsize=16, fontweight="bold", y=1.02)

            # --- Plot Loss ---
            # Removed explicit label='...' to prevent crashes with hue="stage"
            sns.lineplot(data=history, x="epoch_in_stage", y="train_loss", hue="stage", marker='o', ax=axes[0])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_loss", hue="stage", marker='x', linestyle='--', ax=axes[0])
            axes[0].set_title("Loss")
            axes[0].set_xlabel("Epoch in Stage")
            axes[0].set_ylabel("Loss")
            axes[0].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            # --- Plot Accuracy ---
            sns.lineplot(data=history, x="epoch_in_stage", y="train_acc", hue="stage", marker='o', ax=axes[1])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_acc", hue="stage", marker='x', linestyle='--', ax=axes[1])
            axes[1].set_title("Accuracy (%)")
            axes[1].set_xlabel("Epoch in Stage")
            axes[1].set_ylabel("Accuracy (%)")
            axes[1].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[1].get_legend_handles_labels()
            axes[1].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            # --- Plot F1 Macro ---
            sns.lineplot(data=history, x="epoch_in_stage", y="train_f1_macro", hue="stage", marker='o', ax=axes[2])
            sns.lineplot(data=history, x="epoch_in_stage", y="val_f1_macro", hue="stage", marker='x', linestyle='--', ax=axes[2])
            axes[2].set_title("F1 Macro (%)")
            axes[2].set_xlabel("Epoch in Stage")
            axes[2].set_ylabel("F1 Macro (%)")
            axes[2].grid(True, alpha=0.3)
            # Deduplicate the legend
            handles, labels = axes[2].get_legend_handles_labels()
            axes[2].legend(handles=handles[:len(labels)//2], labels=labels[:len(labels)//2])

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plot_fname = f"finetuning_history_{model_name.replace('/', '-').replace(' ', '_')}.png"
            plt.savefig(plot_fname, dpi=150, bbox_inches="tight")
            plt.show()
            print(f"  Saved: {plot_fname}")

        except Exception as e:
            print(f"  ❌ Error plotting history for {model_name}: {e}")

## 📊 4 · Scientific Evaluation

This section evaluates each model using report-friendly **classification metrics only**:
accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC, MCC, Cohen's kappa,
and confusion-derived metrics such as specificity, FPR, FNR, NPV, and row-normalized confusion matrices.

In [ ]:
# ── Evaluation loop ────────────────────────────────────────────
RESULTS = {}

print("Running scientific evaluation on all registered models …\n")
for name, info in MODEL_REGISTRY.items():
    loader_key = info["loader"]
    model = info["model"]

    if loader_key == "32":
        loader = loader_32
    elif loader_key == "32_huy":
        loader = loader_32_huy
    else:
        loader = loader_224

    res = evaluate(model, loader, model_name=name, desc=f"Evaluating {name}")
    RESULTS[name] = res
    RESULTS[name].update({
        "source": info["source"],
        "params_M": info.get("params_M", float("nan")),
        "macs_G": info.get("macs_G", float("nan")),
    })

    print(
        f"  {name:40s}  "
        f"Acc={res['accuracy']:.2f}%  "
        f"F1-macro={res['f1_macro']:.2f}%  "
        f"Spec-macro={res['specificity_macro']:.2f}%  "
        f"ROC-AUC(macro)={res['roc_auc_macro_ovr']:.2f}%  "
        f"MCC={res['mcc']:.4f}"
    )

print("\n✅ Scientific evaluation complete.")

## 📋 5 · Scientific Comparison Table

The main comparison table emphasizes **classification quality** and excludes latency, parameter count, and hardware efficiency metrics from the ranking.

In [ ]:
rows = []
for name, r in RESULTS.items():
    rows.append({
        "Model": name,
        "Accuracy (%)": round(r["accuracy"], 2),
        "Balanced Accuracy (%)": round(r["balanced_accuracy"], 2),
        "Precision Macro (%)": round(r["precision_macro"], 2),
        "Recall Macro (%)": round(r["recall_macro"], 2),
        "F1 Macro (%)": round(r["f1_macro"], 2),
        "F1 Weighted (%)": round(r["f1_weighted"], 2),
        "Specificity Macro (%)": round(r["specificity_macro"], 2),
        "NPV Macro (%)": round(r["npv_macro"], 2),
        "FPR Macro (%)": round(r["fpr_macro"], 2),
        "FNR Macro (%)": round(r["fnr_macro"], 2),
        "ROC AUC Macro OVR (%)": round(r["roc_auc_macro_ovr"], 2),
        "ROC AUC Micro (%)": round(r["roc_auc_micro"], 2),
        "PR AUC Macro (%)": round(r["pr_auc_macro"], 2),
        "MCC": round(r["mcc"], 4),
        "Cohen's Kappa": round(r["cohen_kappa"], 4),
        "Source": r["source"],
    })

df = pd.DataFrame(rows)

# Balanced ranking across the main scientific metrics
df["Rank Accuracy"] = df["Accuracy (%)"].rank(ascending=False, method="min")
df["Rank F1 Macro"] = df["F1 Macro (%)"].rank(ascending=False, method="min")
df["Rank ROC AUC"] = df["ROC AUC Macro OVR (%)"].rank(ascending=False, method="min")
df["Mean Rank"] = df[["Rank Accuracy", "Rank F1 Macro", "Rank ROC AUC"]].mean(axis=1)

df = df.sort_values(
    ["Mean Rank", "F1 Macro (%)", "Accuracy (%)", "ROC AUC Macro OVR (%)"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

df.index += 1
df.insert(0, "Overall Rank", df.index)

metric_cols = [
    "Accuracy (%)", "Balanced Accuracy (%)", "Precision Macro (%)", "Recall Macro (%)",
    "F1 Macro (%)", "F1 Weighted (%)", "Specificity Macro (%)", "NPV Macro (%)",
    "FPR Macro (%)", "FNR Macro (%)", "ROC AUC Macro OVR (%)", "ROC AUC Micro (%)",
    "PR AUC Macro (%)", "MCC", "Cohen's Kappa"
]

styled = (
    df.style
      .background_gradient(
          subset=[
              "Accuracy (%)", "Balanced Accuracy (%)", "Precision Macro (%)", "Recall Macro (%)",
              "F1 Macro (%)", "F1 Weighted (%)", "Specificity Macro (%)", "NPV Macro (%)",
              "ROC AUC Macro OVR (%)", "ROC AUC Micro (%)", "PR AUC Macro (%)"
          ],
          cmap="RdYlGn"
      )
      .background_gradient(subset=["FPR Macro (%)", "FNR Macro (%)"], cmap="RdYlGn_r")
      .background_gradient(subset=["MCC", "Cohen's Kappa"], cmap="RdYlGn")
      .background_gradient(subset=["Mean Rank"], cmap="RdYlGn_r")
      .format({
          "Accuracy (%)": "{:.2f}",
          "Balanced Accuracy (%)": "{:.2f}",
          "Precision Macro (%)": "{:.2f}",
          "Recall Macro (%)": "{:.2f}",
          "F1 Macro (%)": "{:.2f}",
          "F1 Weighted (%)": "{:.2f}",
          "Specificity Macro (%)": "{:.2f}",
          "NPV Macro (%)": "{:.2f}",
          "FPR Macro (%)": "{:.2f}",
          "FNR Macro (%)": "{:.2f}",
          "ROC AUC Macro OVR (%)": "{:.2f}",
          "ROC AUC Micro (%)": "{:.2f}",
          "PR AUC Macro (%)": "{:.2f}",
          "MCC": "{:.4f}",
          "Cohen's Kappa": "{:.4f}",
          "Mean Rank": "{:.2f}",
      })
      .set_caption(
          "CIFAR-10 Scientific Comparison "
          "(ranking based on Accuracy, Macro-F1, and Macro ROC-AUC)"
      )
)

display(styled)
df.to_csv("cifar10_scientific_report_metrics.csv", index=False)
print("Saved: cifar10_scientific_report_metrics.csv")

## 📈 6 · Visualizations

These plots focus on **classification evidence for a scientific report** rather than speed or resource efficiency.

In [ ]:
plot_df = df.sort_values("Mean Rank", ascending=True).reset_index(drop=True)
models = plot_df["Model"].tolist()
metric_specs = [
    ("Accuracy (%)", 95),
    ("F1 Macro (%)", 95),
    ("ROC AUC Macro OVR (%)", 99),
    ("MCC", 0.90),
]

fig, axes = plt.subplots(2, 2, figsize=(18, max(8, len(plot_df) * 0.42)))
fig.suptitle("Scientific Comparison Across Core Classification Metrics", fontsize=16, fontweight="bold", y=1.02)

for ax, (metric, baseline) in zip(axes.flat, metric_specs):
    vals = plot_df[metric].tolist()
    bars = ax.barh(range(len(plot_df)), vals, color=PALETTE[:len(plot_df)], edgecolor="white", linewidth=0.5)
    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(models, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    if metric != "MCC":
        ax.set_xlim(max(0, min(vals) - 2.0), min(100.2, max(vals) + 1.5))
        ax.axvline(baseline, linestyle="--", linewidth=1, color="red", alpha=0.5)
    else:
        ax.set_xlim(max(-0.05, min(vals) - 0.05), min(1.02, max(vals) + 0.03))
        ax.axvline(baseline, linestyle="--", linewidth=1, color="red", alpha=0.5)
    for i, val in enumerate(vals):
        fmt = "{:.4f}" if metric == "MCC" else "{:.2f}"
        ax.text(val + (0.01 if metric == "MCC" else 0.10), i, fmt.format(val), va="center", fontsize=8.5)

plt.tight_layout()
plt.savefig("scientific_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.1 Confusion Matrices *(All Models, Ordered by Mean Rank)*

To keep the notebook readable, the confusion matrices are shown in ranked batches.
Each panel is row-normalized, so values represent the percentage of samples from a true class
that were assigned to each predicted class.

In [ ]:
ordered_models = df["Model"].tolist()
chunk_size = 8
n_cols = 4

for part_idx, start in enumerate(range(0, len(ordered_models), chunk_size), start=1):
    chunk = ordered_models[start:start + chunk_size]
    n_rows = int(math.ceil(len(chunk) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 4.2 * n_rows))
    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)
    fig.suptitle(
        f"Row-Normalized Confusion Matrices – Models Ranked {start + 1} to {start + len(chunk)}",
        fontsize=15,
        fontweight="bold",
    )

    for ax, model_name in zip(axes.flat, chunk):
        cm_norm = RESULTS[model_name]["confusion_matrix_norm"]
        acc = RESULTS[model_name]["accuracy"]
        f1m = RESULTS[model_name]["f1_macro"]

        sns.heatmap(
            cm_norm,
            annot=False,
            fmt=".1f",
            ax=ax,
            xticklabels=CIFAR10_CLASSES,
            yticklabels=CIFAR10_CLASSES,
            cmap="Blues",
            linewidths=0.2,
            linecolor="gray",
            cbar=False,
            vmin=0,
            vmax=100,
        )
        ax.set_title(
            f"{model_name}\nAcc={acc:.2f}% | F1-macro={f1m:.2f}%",
            fontsize=10.5,
            fontweight="bold",
        )
        ax.set_xlabel("Predicted", fontsize=9)
        ax.set_ylabel("True", fontsize=9)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.tick_params(axis="y", rotation=0, labelsize=8)

    for ax in axes.flat[len(chunk):]:
        ax.axis("off")

    plt.tight_layout()
    out_name = f"confusion_matrices_all_models_part{part_idx}.png"
    plt.savefig(out_name, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_name}")

### 6.2 Confusion-Derived Metrics *(All Models)*

The next tables expose confusion-based diagnostics for every model:
- **macro summaries** across classes
- **per-class TP / FP / FN / TN, sensitivity, specificity, FPR, FNR, precision, and NPV**

In [ ]:
confusion_summary_rows = []
confusion_long_rows = []

for model_name in df["Model"]:
    r = RESULTS[model_name]
    top_conf = r["top_confusions"]
    largest_conf = (
        f"{top_conf.iloc[0]['True Class']} → {top_conf.iloc[0]['Predicted As']} ({top_conf.iloc[0]['Rate (%)']:.2f}%)"
        if len(top_conf) else "None"
    )

    confusion_summary_rows.append({
        "Model": model_name,
        "Specificity Macro (%)": r["specificity_macro"],
        "NPV Macro (%)": r["npv_macro"],
        "FPR Macro (%)": r["fpr_macro"],
        "FNR Macro (%)": r["fnr_macro"],
        "Largest Confusion": largest_conf,
    })

    tmp = r["confusion_metrics"].copy()
    tmp.insert(0, "Model", model_name)
    confusion_long_rows.append(tmp)

confusion_summary_df = pd.DataFrame(confusion_summary_rows)
confusion_metrics_all_df = pd.concat(confusion_long_rows, ignore_index=True)

display(
    confusion_summary_df.style.format({
        "Specificity Macro (%)": "{:.2f}",
        "NPV Macro (%)": "{:.2f}",
        "FPR Macro (%)": "{:.2f}",
        "FNR Macro (%)": "{:.2f}",
    })
)

display(
    confusion_metrics_all_df
    .set_index(["Model", "Class"])
    .style
    .format({
        "Precision (%)": "{:.2f}",
        "Recall / Sensitivity (%)": "{:.2f}",
        "Specificity (%)": "{:.2f}",
        "NPV (%)": "{:.2f}",
        "FPR (%)": "{:.2f}",
        "FNR (%)": "{:.2f}",
    })
)

confusion_summary_df.to_csv("cifar10_confusion_summary_all_models.csv", index=False)
confusion_metrics_all_df.to_csv("cifar10_confusion_per_class_all_models.csv", index=False)
print("Saved: cifar10_confusion_summary_all_models.csv")
print("Saved: cifar10_confusion_per_class_all_models.csv")

### 6.3 Per-Class F1 Heatmap

In [ ]:
per_class_f1 = {}
for name, r in RESULTS.items():
    tmp = r["per_class_metrics"].set_index("Class")["F1 Score (%)"]
    per_class_f1[name] = [tmp[c] for c in CIFAR10_CLASSES]

pca_df = pd.DataFrame(per_class_f1, index=CIFAR10_CLASSES).T
pca_df = pca_df.loc[df["Model"]]

fig, ax = plt.subplots(figsize=(14, max(6, len(pca_df) * 0.5)))
sns.heatmap(
    pca_df,
    annot=True,
    fmt=".1f",
    ax=ax,
    cmap="RdYlGn",
    linewidths=0.3,
    linecolor="gray",
    cbar_kws={"label": "Per-Class F1 (%)"},
    vmin=max(0, np.nanmin(pca_df.values) - 5),
    vmax=100,
    annot_kws={"size": 8},
)
ax.set_title("Per-Class F1 Heatmap (rows sorted by overall mean rank)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("CIFAR-10 Class", fontsize=11)
ax.set_ylabel("Model", fontsize=11)
ax.tick_params(axis="x", rotation=30, labelsize=10)
ax.tick_params(axis="y", rotation=0, labelsize=10)

plt.tight_layout()
plt.savefig("per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.4 Micro-Average ROC Curves *(Top-5 Models by Mean Rank)*

In [ ]:
top5_models = df.nsmallest(5, "Mean Rank")["Model"].tolist()

fig, ax = plt.subplots(figsize=(10, 8))
for i, model_name in enumerate(top5_models):
    fpr, tpr = RESULTS[model_name]["roc_curve_micro"]
    roc_auc_micro = RESULTS[model_name]["roc_auc_micro"]
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={roc_auc_micro:.2f}%)", color=PALETTE[i])

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.2, label="Random baseline")
ax.set_title("Micro-Average ROC Curves – Top-5 Models", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("roc_curves_top5.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.5 Top-8 Model Comparison on Report Metrics

In [ ]:
top8 = df.nsmallest(min(8, len(df)), "Mean Rank").copy()
plot_metrics = ["Accuracy (%)", "Balanced Accuracy (%)", "F1 Macro (%)", "ROC AUC Macro OVR (%)"]
plot_long = top8.melt(id_vars=["Model"], value_vars=plot_metrics, var_name="Metric", value_name="Value")

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=plot_long, x="Model", y="Value", hue="Metric", ax=ax)
ax.set_title("Top-8 Models Across Core Report Metrics", fontsize=14, fontweight="bold")
ax.set_ylabel("Score (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35, labelsize=9)
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig("top8_report_metric_grouped_bars.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.6 Per-Class Precision, Recall, Specificity, and F1 for the Best-Ranked Model

In [ ]:
best_model_name = df.iloc[0]["Model"]
best_per_class = RESULTS[best_model_name]["per_class_metrics"].set_index("Class")[
    ["Precision (%)", "Recall / Sensitivity (%)", "Specificity (%)", "F1 Score (%)"]
]

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    best_per_class,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.3,
    linecolor="gray",
    cbar_kws={"label": "Score (%)"},
    vmin=max(0, np.nanmin(best_per_class.values) - 5),
    vmax=100,
    annot_kws={"size": 9},
    ax=ax,
)
ax.set_title(f"Per-Class Report Metrics – {best_model_name}", fontsize=14, fontweight="bold")
ax.set_xlabel("Metric")
ax.set_ylabel("Class")
ax.tick_params(axis="x", rotation=20)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("best_model_per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

display(RESULTS[best_model_name]["per_class_metrics"].style.format({
    "Precision (%)": "{:.2f}",
    "Recall / Sensitivity (%)": "{:.2f}",
    "Specificity (%)": "{:.2f}",
    "F1 Score (%)": "{:.2f}",
}))

### 6.7 Most Frequent Misclassification Pairs

In [ ]:
conf_rows = []
for model_name in df.nsmallest(min(5, len(df)), "Mean Rank")["Model"]:
    tmp = RESULTS[model_name]["top_confusions"].copy()
    tmp["Model"] = model_name
    conf_rows.append(tmp)

if conf_rows:
    conf_df = pd.concat(conf_rows, ignore_index=True)
    agg_conf = (
        conf_df.groupby(["True Class", "Predicted As"], as_index=False)
               .agg({"Rate (%)": "mean", "Count": "sum"})
               .sort_values(["Rate (%)", "Count"], ascending=False)
               .head(12)
    )
else:
    agg_conf = pd.DataFrame(columns=["True Class", "Predicted As", "Rate (%)", "Count"])

display(agg_conf.style.format({"Rate (%)": "{:.2f}", "Count": "{:d}"}))

if len(agg_conf):
    labels = [f"{r['True Class']} → {r['Predicted As']}" for _, r in agg_conf.iterrows()]
    fig, ax = plt.subplots(figsize=(11, max(5, len(agg_conf) * 0.45)))
    bars = ax.barh(labels, agg_conf["Rate (%)"], color=PALETTE[:len(agg_conf)], edgecolor="white", linewidth=0.5)
    ax.invert_yaxis()
    ax.set_xlabel("Average confusion rate (%) across top-ranked models")
    ax.set_title("Dominant Error Modes", fontsize=14, fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    for bar, val in zip(bars, agg_conf["Rate (%)"]):
        ax.text(val + 0.08, bar.get_y() + bar.get_height()/2, f"{val:.2f}%", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig("dominant_error_modes.png", dpi=150, bbox_inches="tight")
    plt.show()

### 6.8 Class Difficulty Analysis *(Mean Per-Class F1 Across All Models)*

In [ ]:
print("=" * 78)
print("  CIFAR-10 BENCHMARK — SCIENTIFIC CLASSIFICATION SUMMARY")
print("=" * 78)

best_overall = df.iloc[0]
best_acc = df.sort_values("Accuracy (%)", ascending=False).iloc[0]
best_f1 = df.sort_values("F1 Macro (%)", ascending=False).iloc[0]
best_auc = df.sort_values("ROC AUC Macro OVR (%)", ascending=False).iloc[0]
best_mcc = df.sort_values("MCC", ascending=False).iloc[0]
best_spec = df.sort_values("Specificity Macro (%)", ascending=False).iloc[0]

print(f"\n  🥇 Best Overall (mean rank) : {best_overall['Model']:30s}  Mean Rank={best_overall['Mean Rank']:.2f}")
print(f"  🎯 Highest Accuracy        : {best_acc['Model']:30s}  {best_acc['Accuracy (%)']:.2f}%")
print(f"  📌 Best Macro-F1          : {best_f1['Model']:30s}  {best_f1['F1 Macro (%)']:.2f}%")
print(f"  📈 Best Macro ROC-AUC     : {best_auc['Model']:30s}  {best_auc['ROC AUC Macro OVR (%)']:.2f}%")
print(f"  🛡️ Best Macro Specificity : {best_spec['Model']:30s}  {best_spec['Specificity Macro (%)']:.2f}%")
print(f"  🤝 Best MCC               : {best_mcc['Model']:30s}  {best_mcc['MCC']:.4f}")

print("\n  Final ranking table:")
print(df[[
    "Overall Rank", "Model", "Accuracy (%)", "Balanced Accuracy (%)",
    "Precision Macro (%)", "Recall Macro (%)", "F1 Macro (%)",
    "Specificity Macro (%)", "FPR Macro (%)", "FNR Macro (%)",
    "ROC AUC Macro OVR (%)", "PR AUC Macro (%)", "MCC", "Cohen's Kappa", "Mean Rank"
]].to_string(index=False))

print("\n  Note on fallback fine-tuning:")
print("   Requested architectures without public CIFAR-10 checkpoints use a")
print("   validation-selected transfer-learning fallback (updated safer protocol):")
print("   multi-epoch head warm-up + low-LR full-network fine-tuning with early stopping.")

## 🏆 7 · Final Summary & Key Findings

In [ ]:
print("=" * 78)
print("  CIFAR-10 BENCHMARK — SCIENTIFIC CLASSIFICATION SUMMARY")
print("=" * 78)

best_overall = df.iloc[0]
best_acc     = df.sort_values("Accuracy (%)", ascending=False).iloc[0]
best_f1      = df.sort_values("F1 Macro (%)", ascending=False).iloc[0]
best_auc     = df.sort_values("ROC AUC Macro OVR (%)", ascending=False).iloc[0]
best_mcc     = df.sort_values("MCC", ascending=False).iloc[0]

print(f"\n  🥇 Best Overall (mean rank) : {best_overall['Model']:30s}  Mean Rank={best_overall['Mean Rank']:.2f}")
print(f"  🎯 Highest Accuracy        : {best_acc['Model']:30s}  {best_acc['Accuracy (%)']:.2f}%")
print(f"  📌 Best Macro-F1          : {best_f1['Model']:30s}  {best_f1['F1 Macro (%)']:.2f}%")
print(f"  📈 Best Macro ROC-AUC     : {best_auc['Model']:30s}  {best_auc['ROC AUC Macro OVR (%)']:.2f}%")
print(f"  🤝 Best MCC               : {best_mcc['Model']:30s}  {best_mcc['MCC']:.4f}")

print("\n  Final ranking table:")
print(df[[
    "Overall Rank", "Model", "Accuracy (%)", "Balanced Accuracy (%)",
    "Precision Macro (%)", "Recall Macro (%)", "F1 Macro (%)",
    "ROC AUC Macro OVR (%)", "PR AUC Macro (%)", "MCC", "Cohen's Kappa", "Mean Rank"
]].to_string(index=False))

print(f"\n  Detailed classification report for the best-ranked model ({best_model_name}):")
print(RESULTS[best_model_name]["class_report"])

print("\n  Saved artifacts:")
for fname in [
    "cifar10_scientific_report_metrics.csv",
    "scientific_metric_comparison.png",
    "confusion_matrices_top6.png",
    "per_class_f1_heatmap.png",
    "roc_curves_top5.png",
    "top8_report_metric_grouped_bars.png",
    "best_model_per_class_metrics.png",
    "dominant_error_modes.png",
    "class_difficulty_f1.png",
]:
    if Path(fname).exists():
        print(f"    ✅ {fname}")